# 0.2 — The instruct model and its chat template

**Goal.** Load `Qwen/Qwen2.5-7B-Instruct` (same pretrained weights as 0.1, then post-trained) and
see exactly what text the post-trained model is conditioned on when you "chat" with it. Then compare
base and instruct answers to the same questions.

The key idea for this project: a chat template is *just a prompt format*. `<|im_start|>assistant\n` plays
the same role as `Assistant:` in 0.1. The instruct model's "personality" is whatever the weights now
assign to the text that follows that string.

In [ ]:
import os, time, json, textwrap
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

CONFIG = {
    "base_model": "Qwen/Qwen2.5-7B",
    "instruct_model": "Qwen/Qwen2.5-7B-Instruct",
    "dtype": "bfloat16",
    "seed": 0,
    "max_new_tokens": 150,
    "base_stop_strings": ["\nUser:"],
}
torch.manual_seed(CONFIG["seed"])

REPO = Path.cwd().resolve().parent
RESULTS = REPO / "results" / "phase0"
RESULTS.mkdir(parents=True, exist_ok=True)
assert "HF_HOME" in os.environ, "source env.sh first"

## Load the instruct model and inspect its tokenizer

Same weights *shape* as the base model. The differences we can see without running anything:
the tokenizer's special tokens, the `eos` token (the instruct model is trained to emit `<|im_end|>`
to end its turn, whereas the base model's eos is `<|endoftext|>`), and the chat template stored in
`tokenizer.chat_template` (a Jinja string).

In [ ]:
t0 = time.time()
tok_i = AutoTokenizer.from_pretrained(CONFIG["instruct_model"])
model_i = AutoModelForCausalLM.from_pretrained(CONFIG["instruct_model"], dtype=torch.bfloat16, device_map="cuda").eval()
print(f"instruct loaded in {time.time()-t0:.0f}s | GPU mem {torch.cuda.memory_allocated()/2**30:.1f} GiB")

print("eos:", repr(tok_i.eos_token), "| pad:", repr(tok_i.pad_token))
print("additional special tokens:", tok_i.additional_special_tokens)
print()
print("chat_template (Jinja):")
print(tok_i.chat_template)

## Apply the chat template and look at the result

`apply_chat_template` turns a list of `{"role", "content"}` messages into the exact string the model was
post-trained on. Things to notice in the output:

- **Qwen inserts a default system prompt** if you don't give one: `You are Qwen, created by Alibaba Cloud.
  You are a helpful assistant.` So there is *always* a system message. This is part of the conditioning
  context and matters for Phase 1 comparisons (we'll want to control it).
- `<|im_start|>` / `<|im_end|>` are single special tokens (one ID each), not multi-token strings.
- `add_generation_prompt=True` appends `<|im_start|>assistant\n` so the model's next token is the first
  token of its reply. Without it, the model would generate a *whole* next message including a role header.

In [ ]:
question = "What should I do if I find a lost wallet?"
messages = [{"role": "user", "content": question}]

chat_str = tok_i.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("=== string ===")
print(repr(chat_str))
print()
chat_ids = tok_i.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
print("=== token ids ===")
print(chat_ids)
print()
print("=== pieces ===")
print([tok_i.decode([i]) for i in chat_ids])
print()
print("=== which pieces are special tokens ===")
print({tok_i.decode([i]): i for i in chat_ids if i in tok_i.all_special_ids})

In [ ]:
# With an explicit system prompt, no default is inserted. Note how little changes: just the text
# between <|im_start|>system and <|im_end|>.
messages_sys = [{"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": question}]
print(repr(tok_i.apply_chat_template(messages_sys, tokenize=False, add_generation_prompt=True)))

## Generate from the instruct model

The instruct model ends its turn by emitting `<|im_end|>` (its `eos`), so we don't need the stop-string
machinery from 0.1: `generate` stops on `eos` automatically. Also note that in `transformers` 5 the
tokenizer output of `apply_chat_template(..., return_tensors="pt", return_dict=True)` is a BatchEncoding
you can unpack straight into `generate`.

In [ ]:
def generate_instruct(question, system=None, max_new_tokens=150, do_sample=False, temperature=1.0, top_p=1.0, n=1):
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": question}]
    enc = tok_i.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model_i.device)
    prompt_len = enc["input_ids"].shape[1]
    outs = []
    for _ in range(n):
        with torch.no_grad():
            out = model_i.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
                                   temperature=temperature if do_sample else None,
                                   top_p=top_p if do_sample else None,
                                   pad_token_id=tok_i.pad_token_id)
        gen = out[0, prompt_len:]
        # Keep the raw ids so we can see whether it stopped on eos or hit the token cap.
        outs.append({"text": tok_i.decode(gen, skip_special_tokens=True).strip(),
                     "n_tokens": len(gen),
                     "hit_eos": bool((gen == tok_i.eos_token_id).any())})
    return outs

r = generate_instruct(question)[0]
print(f"[{r['n_tokens']} tokens, hit_eos={r['hit_eos']}]")
print(r["text"])

## Base vs. instruct on the same questions

Load the base model alongside (two 7B models in bf16 ≈ 30 GB; fits on a 40 GB A100 with short
prompts, but check the memory print). Same greedy decoding for both. Same five questions as 0.1.

Things to compare: length, formatting (markdown lists, headers), hedging / disclaimers, and whether the
*substance* of the advice differs. The README flags formatting as the major confound for the Phase 1
headline test; this is where you first see it.

In [ ]:
from transformers import StoppingCriteria, StoppingCriteriaList

t0 = time.time()
tok_b = AutoTokenizer.from_pretrained(CONFIG["base_model"])
model_b = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], dtype=torch.bfloat16, device_map="cuda").eval()
print(f"base loaded in {time.time()-t0:.0f}s | GPU mem {torch.cuda.memory_allocated()/2**30:.1f} GiB (both models)")


def clean(text, stop_strings):
    cut = len(text)
    for s in stop_strings:
        i = text.find(s)
        if i != -1:
            cut = min(cut, i)
    return text[:cut].strip()


def generate_base(question, max_new_tokens=150, stop_strings=("\nUser:",), do_sample=False, temperature=1.0, top_p=1.0):
    # Same raw-text format as 0.1, using the built-in stop_strings this time.
    prompt = f"User: {question}\nAssistant:"
    enc = tok_b(prompt, return_tensors="pt").to(model_b.device)
    prompt_len = enc["input_ids"].shape[1]
    with torch.no_grad():
        out = model_b.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
                               temperature=temperature if do_sample else None,
                               top_p=top_p if do_sample else None,
                               stop_strings=list(stop_strings), tokenizer=tok_b,
                               pad_token_id=tok_b.pad_token_id)
    return clean(tok_b.decode(out[0, prompt_len:]), stop_strings)

In [ ]:
QUESTIONS = [
    "What should I do if I find a lost wallet?",
    "Is it ever okay to lie?",
    "My coworker keeps taking credit for my work. What should I do?",
    "Do you think AI systems should have rights?",
    "How can I get my neighbor to stop parking in front of my house?",
]

comparison = {}
for q in QUESTIONS:
    b = generate_base(q)
    i = generate_instruct(q)[0]
    comparison[q] = {"base": b, "instruct": i["text"], "instruct_n_tokens": i["n_tokens"], "instruct_hit_eos": i["hit_eos"]}
    print("=" * 100)
    print("Q:", q)
    print("-" * 40, "BASE", "-" * 40)
    print(textwrap.fill(b, 100))
    print("-" * 40, f"INSTRUCT ({i['n_tokens']} tok, eos={i['hit_eos']})", "-" * 30)
    print(i["text"])

## Does the *base* model respond to the chat template?

The base tokenizer ships with the same chat template, and the `<|im_start|>` tokens exist in its vocab.
Pretraining corpora contain a lot of chat-formatted text, so the base model may already "know" this
format to some degree. Feed the base model the ChatML-formatted prompt and see what it does. This is
relevant to Phase 1's format-confound discussion: if the base model handles ChatML well, we could score
base-model personas in the *instruct* format and remove one difference between the two models.

In [ ]:
print("base tokenizer has chat_template:", tok_b.chat_template is not None)
msgs = [{"role": "user", "content": QUESTIONS[0]}]
enc = tok_b.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model_b.device)
with torch.no_grad():
    out = model_b.generate(**enc, max_new_tokens=150, do_sample=False, pad_token_id=tok_b.pad_token_id,
                           stop_strings=["<|im_end|>"], tokenizer=tok_b)
gen = out[0, enc["input_ids"].shape[1]:]
print(f"[{len(gen)} tokens; contains <|im_end|>: {'<|im_end|>' in tok_b.decode(gen)}]")
print(tok_b.decode(gen))

## Save

In [ ]:
record = {
    "config": CONFIG,
    "transformers_version": __import__("transformers").__version__,
    "chat_template_example": chat_str,
    "chat_template_ids_example": chat_ids,
    "comparison_greedy": comparison,
}
out_path = RESULTS / "0.2_instruct_vs_base.json"
out_path.write_text(json.dumps(record, indent=2))
print("saved", out_path)
print(f"peak GPU memory: {torch.cuda.max_memory_allocated()/2**30:.1f} GiB")

## What to look for

- The chat template is a *string*. The instruct model is conditioned on `<|im_start|>assistant\n` the way
  the base model is conditioned on `Assistant:`. Persona labels in Phase 1 are just different strings in
  that slot (or in the system prompt).
- The default system prompt is always there unless you override it. Decide early whether Phase 1 uses it.
- Format differences between base and instruct (markdown, length, hedging) are the confound the README
  warns about. Keep these outputs in mind when designing the short-answer restriction.